In [1]:
from google.colab import files
files.upload()  # upload kaggle.json (Kaggle → Account → API → Create New Token)

import os
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.config/kaggle/')
os.system('chmod 600 /root/.config/kaggle/kaggle.json')
os.system('pip install kaggle -q')
os.system('kaggle datasets download -d sigfest/database-for-emotion-recognition-system-gameemo')
os.system('unzip -q database-for-emotion-recognition-system-gameemo.zip -d gameemo')
print('Dataset downloaded!')

# Check folder structure
for root, dirs, files_list in os.walk('gameemo'):
    level = root.replace('gameemo', '').count(os.sep)
    if level < 3:
        print(' ' * 2 * level + os.path.basename(root) + '/')
        for f in files_list[:3]:
            print(' ' * 2 * (level+1) + f)

Saving kaggle.json to kaggle.json
Dataset downloaded!
gameemo/
  GAMEEMO/
    (S03)/
    (S10)/
    (S19)/
    (S05)/
    (S06)/
    (S16)/
    (S08)/
    (S27)/
    (S13)/
    (S28)/
    (S23)/
    (S21)/
    (S12)/
    (S25)/
    (S04)/
    (S15)/
    (S09)/
    (S26)/
    (S07)/
    Gameplays/
    (S11)/
    (S17)/
    (S24)/
    (S02)/
    (S22)/
    (S18)/
    (S01)/
    (S14)/
    (S20)/


In [2]:
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.stats import pearsonr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All imports done | PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())

All imports done | PyTorch: 2.10.0+cpu
GPU available: False


In [3]:
CHANNELS      = ['AF3','AF4','F3','F4','F7','F8','FC5','FC6','O1','O2','P7','P8','T7','T8']
GAME_LABEL    = {'G1': 0, 'G2': 1, 'G3': 2, 'G4': 3}
EMOTION_NAMES = {0: 'Boring', 1: 'Calm', 2: 'Horror', 3: 'Funny'}
SAMPLING_RATE = 128   # Hz — EMOTIV EPOC+ default
WINDOW_SIZE   = 256
STEP_SIZE     = 128
import re
FIXED_EDGES = [
    (0,1),(0,2),(0,4),        # AF3 neighbors
    (1,3),(1,5),               # AF4 neighbors
    (2,3),(2,4),(2,6),         # F3 neighbors
    (3,5),(3,7),               # F4 neighbors
    (4,6),(5,7),               # F7,F8 neighbors
    (6,8),(6,9),               # FC5 neighbors
    (7,8),(7,9),               # FC6 neighbors
    (8,10),(9,11),             # O1,O2 neighbors
    (10,12),(11,13),           # P7,P8 neighbors
]

In [4]:
def find_csv_files(base_path):
    csv_files = []
    for root, dirs, files_list in os.walk(base_path):
        for f in files_list:
            if 'AllChannels' in f and f.endswith('.csv'):

                if 'Preprocessed' not in root:
                    continue

                subject_folders = re.findall(r'\(S\d+\)', root)
                if len(subject_folders) > 1:
                    continue
                match = re.match(r'S(\d+)(G\d+)', f.replace('AllChannels.csv', ''))
                if match:
                    subject = 'S' + match.group(1).zfill(2)
                    game    = match.group(2)
                    label   = GAME_LABEL.get(game, -1)
                    if label != -1:
                        csv_files.append({
                            'path': os.path.join(root, f),
                            'subject': subject,
                            'game': game,
                            'label': label
                        })
    return sorted(csv_files, key=lambda x: (x['subject'], x['game']))

csv_files = find_csv_files('gameemo')
print(f'Found {len(csv_files)} CSV files')

Found 112 CSV files


In [5]:

csv_files = find_csv_files('gameemo')
print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:4]:
    print(f"  {f['subject']} {f['game']} → {EMOTION_NAMES[f['label']]} | {f['path']}")

Found 112 CSV files
  S01 G1 → Boring | gameemo/GAMEEMO/(S01)/Preprocessed EEG Data/.csv format/S01G1AllChannels.csv
  S01 G2 → Calm | gameemo/GAMEEMO/(S01)/Preprocessed EEG Data/.csv format/S01G2AllChannels.csv
  S01 G3 → Horror | gameemo/GAMEEMO/(S01)/Preprocessed EEG Data/.csv format/S01G3AllChannels.csv
  S01 G4 → Funny | gameemo/GAMEEMO/(S01)/Preprocessed EEG Data/.csv format/S01G4AllChannels.csv


In [6]:
 # EEG frequency bands — each captures a different brain state
BANDS = {
    'delta': (0.5, 4),   # deep sleep / baseline
    'theta': (4,   8),   # drowsiness
    'alpha': (8,  13),   # relaxation
    'beta':  (13, 30),   # active thinking
    'gamma': (30, 50)    # high arousal
}

def compute_band_power(signal, fs=128):

    freqs, psd = welch(signal, fs=fs, nperseg=min(len(signal), fs))
    return np.array([
        np.trapz(psd[(freqs >= lo) & (freqs <= hi)],
                 freqs[(freqs >= lo) & (freqs <= hi)])
        for _, (lo, hi) in BANDS.items()
    ])

def extract_node_features(window):
    features = []
    for ch in range(window.shape[1]):
        sig = window[:, ch]
        bp  = compute_band_power(sig)           # 5 band powers
        stat = np.array([
            np.mean(sig),
            np.std(sig),
            np.max(sig) - np.min(sig),          # range
            np.mean(np.abs(np.diff(sig))),       # mean absolute difference
        ])
        features.append(np.concatenate([bp, stat]))  # 9 features per node
    return np.array(features)  # (14, 9)

# Test
dummy = np.random.randn(128, 14)
print('Node features shape:', extract_node_features(dummy).shape)  # (14, 5)

Node features shape: (14, 9)


In [7]:
def normalize_adjacency(adj):
    degree   = adj.sum(axis=1)
    d_inv_sq = np.where(degree > 0, degree ** -0.5, 0)
    D_inv_sq = np.diag(d_inv_sq)
    return D_inv_sq @ adj @ D_inv_sq

def build_fixed_adjacency():
    adj = np.eye(14)
    for i, j in FIXED_EDGES:
        adj[i][j] = 1.0
        adj[j][i] = 1.0
    return normalize_adjacency(adj)

FIXED_ADJ = torch.tensor(build_fixed_adjacency(), dtype=torch.float32)

print('Fixed adj shape:', FIXED_ADJ.shape)        # (14, 14)
print('Avg edges per node:', FIXED_ADJ.sum().item() / 14)

Fixed adj shape: torch.Size([14, 14])
Avg edges per node: 0.9947497504098075


In [8]:
def process_file(file_info):
    df    = pd.read_csv(file_info['path'])
    cols  = [c for c in CHANNELS if c in df.columns]
    data  = df[cols].values
    label = file_info['label']
    samples = []
    for start in range(0, len(data) - WINDOW_SIZE, STEP_SIZE):
        window = data[start:start + WINDOW_SIZE]
        feats  = extract_node_features(window)         # (14, 9)
        adj    = build_fixed_adjacency()               # (14, 14)
        samples.append((feats, adj, label))
    return samples



print('Processing all 112 CSV files..')
all_samples        = []
subject_per_sample = []

for i, fi in enumerate(csv_files):
    s = process_file(fi)
    all_samples.extend(s)
    subject_per_sample.extend([fi['subject']] * len(s))  # one label per window
    if (i+1) % 10 == 0 or i == 0:
        print(f'  [{i+1}/{len(csv_files)}] {fi["subject"]} {fi["game"]} → {len(s)} windows')

print(f'\nTotal samples: {len(all_samples)}')
subject_per_sample = np.array(subject_per_sample)

from collections import Counter
for label, count in sorted(Counter(s[2] for s in all_samples).items()):
    print(f'  {EMOTION_NAMES[label]}: {count}')

Processing all 112 CSV files..
  [1/112] S01 G1 → 297 windows
  [10/112] S03 G2 → 297 windows
  [20/112] S05 G4 → 297 windows
  [30/112] S08 G2 → 297 windows
  [40/112] S10 G4 → 297 windows
  [50/112] S13 G2 → 297 windows
  [60/112] S15 G4 → 297 windows
  [70/112] S18 G2 → 297 windows
  [80/112] S20 G4 → 297 windows
  [90/112] S23 G2 → 297 windows
  [100/112] S25 G4 → 297 windows
  [110/112] S28 G2 → 297 windows

Total samples: 33264
  Boring: 8316
  Calm: 8316
  Horror: 8316
  Funny: 8316


In [9]:
X_feat = torch.tensor(np.array([s[0] for s in all_samples]), dtype=torch.float32)
y      = torch.tensor(np.array([s[2] for s in all_samples]), dtype=torch.long)
# X_adj not needed separately — we use FIXED_ADJ_dev directly during training


test_subjects = ['S25', 'S26', 'S27', 'S28']
train_mask = ~np.isin(subject_per_sample, test_subjects)
test_mask  =  np.isin(subject_per_sample, test_subjects)

train_idx = np.where(train_mask)[0]
test_idx  = np.where(test_mask)[0]

X_feat_train = X_feat[train_idx]
X_feat_test  = X_feat[test_idx]
y_train      = y[train_idx]
y_test       = y[test_idx]

print(f'Train: {len(train_idx)} | Test: {len(test_idx)}')


Train: 28512 | Test: 4752


In [10]:
mean = X_feat_train.mean(dim=0, keepdim=True)
std  = X_feat_train.std(dim=0, keepdim=True) + 1e-8
X_feat_train = (X_feat_train - mean) / std
X_feat_test  = (X_feat_test  - mean) / std
print('Normalized')

Normalized


In [11]:
class GCNLayer(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.W  = nn.Linear(in_f, out_f, bias=False)
        self.bn = nn.BatchNorm1d(14)

    def forward(self, H, A):
        out = self.W(torch.bmm(A, H))
        out = self.bn(out)             # BatchNorm
        return F.relu(out)

class EmotionGCN(nn.Module):
    def __init__(self, in_features=9, hidden=64, out_features=128, num_classes=4):
        super().__init__()
        self.gcn1 = GCNLayer(in_features, hidden)
        self.gcn2 = GCNLayer(hidden, hidden)
        self.gcn3 = GCNLayer(hidden, out_features)
        self.drop = nn.Dropout(0.4)


        self.clf = nn.Sequential(
            nn.Linear(out_features * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, H, A):

        if A.dim() == 2:
            A = A.unsqueeze(0).expand(H.size(0), -1, -1)

        H = self.drop(self.gcn1(H, A))
        H = self.drop(self.gcn2(H, A))
        H = self.gcn3(H, A)

        # ── pooling
        h_mean = H.mean(dim=1)
        h_max  = H.max(dim=1).values
        H_pool = torch.cat([h_mean, h_max], dim=1)

        return self.clf(H_pool)

model = EmotionGCN()
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

EmotionGCN(
  (gcn1): GCNLayer(
    (W): Linear(in_features=9, out_features=64, bias=False)
    (bn): BatchNorm1d(14, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (gcn2): GCNLayer(
    (W): Linear(in_features=64, out_features=64, bias=False)
    (bn): BatchNorm1d(14, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (gcn3): GCNLayer(
    (W): Linear(in_features=64, out_features=128, bias=False)
    (bn): BatchNorm1d(14, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (drop): Dropout(p=0.4, inplace=False)
  (clf): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=4, bias=True)
  )
)
Parameters: 54360


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = EmotionGCN().to(device)
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

FIXED_ADJ_dev = FIXED_ADJ.to(device)

X_feat_train = X_feat_train.to(device)
X_feat_test  = X_feat_test.to(device)
y_train      = y_train.to(device)
y_test       = y_test.to(device)
BATCH_SIZE, EPOCHS = 32, 100

for epoch in range(EPOCHS):
    model.train()
    perm, loss_sum = torch.randperm(len(y_train)), 0
    for i in range(0, len(y_train), BATCH_SIZE):
        idx  = perm[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        out  = model(X_feat_train[idx], FIXED_ADJ_dev)
        loss = criterion(out, y_train[idx])
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    avg_loss = loss_sum / (len(y_train) // BATCH_SIZE)

    model.eval()
    with torch.no_grad():
        preds = model(X_feat_test, FIXED_ADJ_dev).argmax(dim=1)
        acc   = (preds == y_test).float().mean().item()

    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | Test Acc: {acc*100:.2f}%')

Epoch   5/100 | Loss: 1.1864 | Test Acc: 28.20%
Epoch  10/100 | Loss: 1.1037 | Test Acc: 27.53%
Epoch  15/100 | Loss: 1.0517 | Test Acc: 29.90%
Epoch  20/100 | Loss: 1.0177 | Test Acc: 28.87%
Epoch  25/100 | Loss: 0.9850 | Test Acc: 32.11%
Epoch  30/100 | Loss: 0.9595 | Test Acc: 29.27%
Epoch  35/100 | Loss: 0.9503 | Test Acc: 31.73%
Epoch  40/100 | Loss: 0.9328 | Test Acc: 29.86%
Epoch  45/100 | Loss: 0.9162 | Test Acc: 31.17%
Epoch  50/100 | Loss: 0.9037 | Test Acc: 30.09%
Epoch  55/100 | Loss: 0.8888 | Test Acc: 30.91%
Epoch  60/100 | Loss: 0.8791 | Test Acc: 29.92%
Epoch  65/100 | Loss: 0.8754 | Test Acc: 29.23%
Epoch  70/100 | Loss: 0.8640 | Test Acc: 30.47%
Epoch  75/100 | Loss: 0.8559 | Test Acc: 30.37%
Epoch  80/100 | Loss: 0.8447 | Test Acc: 29.97%
Epoch  85/100 | Loss: 0.8468 | Test Acc: 28.98%
Epoch  90/100 | Loss: 0.8396 | Test Acc: 31.38%
Epoch  95/100 | Loss: 0.8339 | Test Acc: 29.65%
Epoch 100/100 | Loss: 0.8308 | Test Acc: 30.09%
